# 02 · Train the court keypoint model on several datasets

The court model finds up to 18 court landmarks in a frame. Four or more give the mapping from the
broadcast image to the court in metres, which turns every player's feet into court x, y. A frame
without a mapping is a frame without positions, so the number that matters most is **how often a
real broadcast frame gets mapped**.

| Source | What it adds | Role |
|---|---|---|
| `reloc2` | public court-keypoint set (fyp-3bwmg/reloc2-den7l v1): hand-labelled, whole court centred in frame | train |
| `gamedet_court`, `broadcast_court` | frames of Basketball Game Detection and Basketball Broadcast, labelled by the current court model only where its keypoints fit the real court geometry | train |
| `gamedet_holdout`, `broadcast_holdout` | whole videos of those two sets, kept back and never labelled | test only |
| `own_court` | your frames from notebook 03, court keypoints corrected in Roboflow | train (weighted x3) |
| `own_court_test` | frames of clips you keep back | test only |

**Why pseudo-labels help.** The public set only shows the full court. The broadcast frames show
the views the pipeline meets (half court, zoomed, panning), but nobody labelled their landmarks.
The current model's keypoints on such a frame are kept only when a homography fits all of them
within 0.25 m and maps each back within 1% of the image width, and the mapping is the right way
round (not a mirror image with the two halves mixed up). The fitted mapping then places **every**
landmark in the frame, including the ones the model missed, and those are what the new model
learns from.

**How leakage is kept out:**

* Whole videos of each broadcast set are held back **before** pseudo-labelling, so the model being
  replaced never labels a frame that is later used to score it. Held-back frames are a test-only
  source, which also pulls any near-copy of them into test.
* Images are tied by video and by look across all datasets, and tied images share a split
  (the same build as notebook 01).
* Zoomed crops are made only from training images, after the split, and inherit it.
* Pseudo-labelled frames are never used to measure accuracy: the held-back broadcast frames are
  scored by mapping coverage (no labels needed) and landmark accuracy is measured only on
  hand-labelled frames.

Landmark order (`courtvision/court.py`):

| idx | landmark | idx | landmark |
|---|---|---|---|
| 0-5 | left baseline: corner, corner-3, lane, lane, corner-3, corner | 10-15 | right baseline (mirrored) |
| 6-7 | half-court line ends | 16-17 | right free-throw line ends |
| 8-9 | left free-throw line ends | | |

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"   # everything that must survive a disconnect lives here
for d in ["models", "runs", "reports", "training_data", "input_videos", "output"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # to test a pull request, put its branch name here, then Runtime > Restart session > Run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models,train]"
import sys
sys.path.insert(0, "/content/courtvision")   # importable now, without restarting the session

## 1 · Download the datasets

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))   # Colab Secrets (key icon). Never paste a key into a cell.

def download(workspace, project, version=None):
    """A Roboflow dataset in YOLO format; version None means the latest."""
    p = rf.workspace(workspace).project(project)
    version = version or max(int(str(v.version).split("/")[-1]) for v in p.versions())
    d = p.version(version).download("yolov8", location=f"/content/datasets/{project}-v{version}")
    print(f"{workspace}/{project} v{version} -> {d.location}")
    return d.location

ROOTS = {
    "reloc2": download("fyp-3bwmg", "reloc2-den7l", 1),
    "gamedet": download("deeplearning-gmyc6", "basketball-game-detection"),
    "broadcast": download("basketball-w376e", "basketball-broadcast"),
}

## 2 · Check the landmark order on the hand labels

The numbers drawn on the court must match the table above (0 at the left baseline's first corner,
and so on). Then every hand-labelled frame is fitted, and the share that maps the same way round
is printed: real views all share one orientation, so it should be close to 100%. That orientation
is what the pseudo-labels and the scores below check against.

In [ ]:
import glob, shutil, cv2, yaml
import numpy as np, pandas as pd
from collections import Counter
from pathlib import Path
from courtvision.court import FLIP_IDX, NBA
from courtvision.training.labels import label_orientation
from courtvision.training.look import show_grid

reloc2_images = glob.glob(f"{ROOTS['reloc2']}/*/images/*")
show_grid(reloc2_images, ["court"], title="reloc2 labels", n=3)

signs = Counter(label_orientation(row, NBA) for lbl in glob.glob(f"{ROOTS['reloc2']}/*/labels/*.txt")
                for row in open(lbl) if row.split())
fitted = sum(v for k, v in signs.items() if k)
SIGN = max((k for k in signs if k), key=signs.get)
print(f"orientation {SIGN:+d} in {signs[SIGN] / fitted:.1%} of {fitted} labelled frames ({signs[None]} had under 4 landmarks)")
assert signs[SIGN] / fitted > 0.9, "labels disagree on orientation: check the landmark order before training"

## 3 · Pseudo-label the broadcast frames

`HOLDOUT_FRAC` of each set's videos are held back first. On the rest, each frame is kept only if
the current model's keypoints pass the geometry checks; the printout says how many frames were
labelled and why the others were rejected (`too_few_keypoints`: zoomed past what the model sees,
`high_error` or `outlier_keypoints`: a landmark in the wrong place, `disagrees_with_geometry`,
`mirrored`: halves mixed up). Rejected frames are simply not used.

In [ ]:
from ultralytics import YOLO
from courtvision.detect import best_court_instance
from courtvision.training.labels import court_row_from_keypoints
from courtvision.training.sources import SPLITS, hold_out_groups, images_in

CURRENT = f"{DRIVE_ROOT}/models/court_keypoints.pt"
HOLDOUT_FRAC = 0.2
POSE_YAML = {"names": {0: "court"}, "kpt_shape": [18, 3]}
teacher = YOLO(CURRENT)

def pseudo_label(name, root):
    """Hold back whole videos (unlabelled, for scoring); label the rest where the geometry checks pass."""
    imgs = [p for s in SPLITS if (Path(root) / s / "images").exists() for p in images_in(Path(root) / s / "images")]
    held = hold_out_groups([p.stem for p in imgs], HOLDOUT_FRAC, seed=0)
    lab, hold = Path(f"/content/datasets/{name}_court"), Path(f"/content/datasets/{name}_holdout")
    shutil.rmtree(lab, ignore_errors=True); shutil.rmtree(hold, ignore_errors=True)
    for d in [lab / "train" / "images", lab / "train" / "labels", hold / "test" / "images"]:
        d.mkdir(parents=True)
    reasons = Counter()
    for img_path, h in zip(imgs, held):
        if h:
            shutil.copy(img_path, hold / "test" / "images" / img_path.name)
            continue
        img = cv2.imread(str(img_path))
        kps = best_court_instance(teacher.predict(img, imgsz=640, conf=0.25, verbose=False)[0])
        row, info = court_row_from_keypoints(kps, NBA, (img.shape[1], img.shape[0]), orientation_sign=SIGN)
        reasons[info["reason"]] += 1
        if row:
            shutil.copy(img_path, lab / "train" / "images" / img_path.name)
            (lab / "train" / "labels" / f"{img_path.stem}.txt").write_text(row + "\n")
    for d in [lab, hold]:
        yaml.safe_dump(POSE_YAML, open(d / "data.yaml", "w"))
    print(f"{name}: {int(held.sum())} frames held back; labelled {reasons['ok']} of {sum(reasons.values())}; {dict(reasons)}")
    return str(lab), str(hold)

PSEUDO = {name: pseudo_label(name, ROOTS[name]) for name in ["gamedet", "broadcast"]}

In [ ]:
# Look at what was labelled: landmarks should sit on the court lines, including ones the model missed.
for name, (lab, hold) in PSEUDO.items():
    show_grid(glob.glob(f"{lab}/train/images/*"), ["court"], title=f"{name}: pseudo-labels",
              save=f"{DRIVE_ROOT}/reports/look_{name}_court.jpg")

## 4 · Build one leakage-free dataset, then add zoomed crops

Same build as notebook 01. Afterwards, two zoomed half-court crops of each `reloc2` **training**
image are added (landmarks outside the crop become invisible), so the full-court public set also
teaches zoomed views. `flip_idx` lets training flip images left-right with the landmarks swapped.

In [ ]:
from courtvision.training.crops import add_zoomed_crops
from courtvision.training.sources import Source, build, class_names, leakage_report, summary

def court_source(name, root, **kw):
    """A one-class pose dataset, whatever its class is called."""
    return Source(name, root, rename={n: "court" for n in class_names(Path(root) / "data.yaml")}, **kw)

TD = f"{DRIVE_ROOT}/training_data"
SOURCES = [court_source("reloc2", ROOTS["reloc2"])]
for name, (lab, hold) in PSEUDO.items():
    SOURCES += [court_source(f"{name}_court", lab), court_source(f"{name}_holdout", hold, role="test")]
if os.path.exists(f"{TD}/own_labelled_court/data.yaml"):
    SOURCES.append(court_source("own_court", f"{TD}/own_labelled_court", weight=3))
if os.path.exists(f"{TD}/own_court_test/data.yaml"):
    SOURCES.append(court_source("own_court_test", f"{TD}/own_court_test", role="test"))

DATA = "/content/datasets/court"
manifest = build(SOURCES, ["court"], DATA, val_frac=0.15, test_frac=0.15, seed=0, kpt_shape=[18, 3], flip_idx=FLIP_IDX)
report = leakage_report(manifest)
print(report)
assert report["components_in_two_splits"] == report["groups_in_two_splits"] == report["test_source_images_not_in_test"] == 0
print("zoomed crops added:", add_zoomed_crops(DATA, ["reloc2"], per_image=2))
manifest = pd.read_csv(f"{DATA}/manifest.csv")
manifest.to_csv(f"{DRIVE_ROOT}/reports/court_manifest.csv", index=False)
show_grid(manifest[manifest.crop_of.notna()].image.tolist(), ["court"], title="zoomed crops", n=3)
summary(manifest, ["court"])

## 5 · Scoring and training helpers

* `mapped_share`: frames whose keypoints give a consistent, right-way-round mapping (no labels needed)
* `keypoints_per_frame`: confident landmarks per frame
* `median_px_error`, `pck_2pct` (share of labelled landmarks found within 2% of the image width),
  on hand-labelled frames only

What each model is scored on: `reloc2` and `own_court` on their test split; `gamedet` and
`broadcast` on their held-back videos; `own_court_test` on all of it.

In [ ]:
from courtvision.config import Config
from courtvision.training.evaluate import court_scores

RUN_TAG = "v1"   # change when the sources change, so an old finished run isn't reused
CFG = Config.load("/content/courtvision/configs/default.yaml")
HAND_LABELLED = {"reloc2", "own_court", "own_court_test"}

def eval_images(where=None):
    """{name: images} to score on; where: optional extra row filter on the manifest."""
    m = manifest[manifest.crop_of.isna() & (where if where is not None else True)]
    sets = {"reloc2": m[(m.source == "reloc2") & (m.split == "test")],
            "own_court": m[(m.source == "own_court") & (m.split == "test")],
            "own_court_test": m[m.source == "own_court_test"]}
    sets.update({name: m[m.source == f"{name}_holdout"] for name in PSEUDO})
    return {k: v.image.tolist() for k, v in sets.items() if len(v)}

def score(weights, by_name, imgsz=640):
    model = YOLO(weights)
    rows = {}
    for name, imgs in by_name.items():
        r = court_scores(model, imgs, NBA, imgsz=imgsz, kp_conf=CFG.court.keypoint_conf, orientation_sign=SIGN)
        if name not in HAND_LABELLED:
            r["median_px_error"] = r["pck_2pct"] = None   # no hand labels there
        rows[name] = r
    return pd.DataFrame(rows).T

def train(name, data_yaml, base, **args):
    """Train, or resume after a disconnect; returns the best weights. Finished runs are reused."""
    run = f"{DRIVE_ROOT}/runs/{name}"
    if not os.path.exists(f"{run}/done"):
        last = f"{run}/weights/last.pt"
        if os.path.exists(last):
            try:
                YOLO(last).train(resume=True)
            except AssertionError as e:   # it had finished; only the marker was missing
                print(e)
        else:
            YOLO(base).train(data=data_yaml, project=f"{DRIVE_ROOT}/runs", name=name, exist_ok=True, **args)
        open(f"{run}/done", "w").close()
    return f"{run}/weights/best.pt"

AUG = dict(fliplr=0.5, mosaic=0.5, degrees=0, perspective=0.0005, scale=0.3)

## 6 · Cross-dataset validation: leave one dataset out

For each training source, a small model trained with everything and one trained without that
source (and anything tied to it). If the model trained without the broadcast pseudo-labels maps
far fewer held-back broadcast frames, those labels are doing their job; if the one trained without
`reloc2` loses accuracy on `reloc2`, the full-court views still need the public set. Set
`RUN_LODO = False` to skip.

In [ ]:
from courtvision.training.sources import lodo_folds

RUN_LODO = True
CV = dict(imgsz=640, epochs=60, patience=15, batch=-1, cos_lr=True, plots=False, **AUG)
EVAL_OF = {"reloc2": "reloc2", "gamedet_court": "gamedet", "broadcast_court": "broadcast", "own_court": "own_court"}
folds = lodo_folds(DATA, min_images=50) if RUN_LODO else {}   # sources with 50+ images
if folds:
    targets = eval_images()
    with_it = score(train(f"court_{RUN_TAG}_cv_all", f"{DATA}/data.yaml", "yolo11s-pose.pt", **CV), targets)
    rows = []
    for held, y in folds.items():
        best = train(f"court_{RUN_TAG}_cv_without_{held}", str(y), "yolo11s-pose.pt", **CV)
        name = EVAL_OF.get(held, held)
        if name in targets:
            rows.append(score(best, {name: targets[name]}))
    cols = ["mapped_share", "keypoints_per_frame", "pck_2pct"]
    lodo = pd.concat({"with": with_it[cols], "without": pd.concat(rows)[cols]}, axis=1).swaplevel(axis=1)[cols]
    lodo.to_csv(f"{DRIVE_ROOT}/reports/court_lodo_{RUN_TAG}.csv")
    display(lodo)

## 7 · Train on everything

In [ ]:
FINAL = dict(imgsz=640, epochs=200, patience=40, batch=-1, cos_lr=True, save_period=10, **AUG)
NEW = train(f"court_{RUN_TAG}_final_yolo11m_pose", f"{DATA}/data.yaml", "yolo11m-pose.pt", **FINAL)

## 8 · New model vs current model

The current court model was trained on `reloc2` with its own random split, so it may have seen
any `reloc2` test image: that row is for reference. The comparison that decides uses frames
neither model trained on and that are not tied to any `reloc2` image.

In [ ]:
reloc2_components = set(manifest[manifest.source == "reloc2"].component)
unseen_by_current = ~manifest.component.isin(reloc2_components)
fair = {k: v for k, v in eval_images(unseen_by_current).items() if k != "reloc2"}
table = pd.concat({"new": score(NEW, fair), "current": score(CURRENT, fair)}).swaplevel().sort_index()
reference = pd.concat({"new": score(NEW, {"reloc2": eval_images()["reloc2"]}),
                       "current": score(CURRENT, {"reloc2": eval_images()["reloc2"]})})
table.to_csv(f"{DRIVE_ROOT}/reports/court_new_vs_current_{RUN_TAG}.csv")
display(table)
display(reference)

## 9 · Keep the new model only if it is better

On every set in the fair comparison the new model must map at least as many frames and find at
least as many landmarks (within 0.01), and on hand-labelled frames be at least as accurate. On
`reloc2` it may not lose more than 0.03 PCK, so full-court views don't get worse. The previous file
is kept as `court_keypoints_prev.pt`.

In [ ]:
from courtvision.training.evaluate import compare

verdict = {}
for name in table.index.get_level_values(0).unique():
    ok, why = compare(table.loc[(name, "new")], table.loc[(name, "current")],
                      higher=["mapped_share", "keypoints_per_frame", "pck_2pct"], lower=["median_px_error"])
    verdict[name] = "ok" if ok else "; ".join(why)
ok, why = compare(reference.loc[("new", "reloc2")], reference.loc[("current", "reloc2")], higher=["pck_2pct"], tol=0.03)
verdict["reloc2 (guard)"] = "ok" if ok else "; ".join(why)
for k, v in verdict.items():
    print(f"{k:16s} {v}")

if os.path.exists(CURRENT) and any(v != "ok" for v in verdict.values()):
    print("\nKept the current court model. The new one is at", NEW)
else:
    if os.path.exists(CURRENT):
        shutil.copy(CURRENT, f"{DRIVE_ROOT}/models/court_keypoints_prev.pt")
    shutil.copy(NEW, CURRENT)
    print("\nSaved", CURRENT)